<!-- source: new + PRZ[19] -->
# Przygotowanie prowadzącego: workspace Premium w dniu warsztatu

**Kto:** prowadzący. **Kiedy:** dzień przed warsztatem i jeszcze raz rano (tylko komórka gotowości). Najdłużej trwa start endpointu i indeksu AI Search.

Dane do repozytorium przygotowuje osobny notebook `scripts/prepare_data_premium.ipynb` (raz, przed commitem danych). Ten notebook stawia wszystko, czego potrzebują **dema** `demo/m1–m6`, i kończy się tabelą „gotowe / niegotowe”.

| Krok | Co | Jak |
|---|---|---|
| 1 | tabele, Volume, chunki jak u uczestników | **Run all** na `00_setup/00_setup` (ten sam notebook co uczestnicy) |
| 2 | endpoint i indeks AI Search, z czekaniem do skutku | komórka poniżej |
| 3 | trzy funkcje UC | **Run all** na `demo/m2_tool_calling` |
| 4 | Genie Agent `Retail Customer Intelligence Assistant` | UI, instrukcja w M4, część 2 |
| 5 | Knowledge Assistant | UI, instrukcja w M3 (demo prowadzącego) |
| 6 | agent `@champion` i Databricks App `sqlday-retail-agent` | komórki prowadzącego w `demo/m5_end_to_end_agent` |
| 7 | dema wzorca Krzysztofa: `pattern/p2_uc_functions_bakehouse`, `pattern/p3_rag_robotics` (parsowanie i indeks robotyki trwają najdłużej) | **Run all** dzień wcześniej |
| 8 | **gotowość**: 14 sprawdzeń | ostatnia komórka |

Po warsztacie uruchom `02_trainer_teardown`.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new
import os
import time
from pathlib import Path

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
USERNAME = spark.sql("SELECT current_user()").first()[0]
DATA_DIR = Path(os.getcwd()).parent / "data"
APP_NAME = "sqlday-retail-agent"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.retail_customer_agent"
KA_ENDPOINT = ""  # po utworzeniu Knowledge Assistant wpisz nazwę jego endpointu (karta agenta)
print(f"Prowadzący: {USERNAME} | workspace: {HOST}")

<!-- source: new -->
## 1. Dane jak u uczestników

Otwórz `00_setup/00_setup` i uruchom **Run all**. Prowadzący pracuje na tych samych obiektach i tych samych liczbach co uczestnicy, dzięki czemu wyniki demo i labu da się porównać 1:1. Komórka poniżej tylko sprawdza efekt.

In [ ]:
# source: new
expected = {GOLD_TABLE: 28_813, CHUNKS_TABLE: None}
for table, rows in expected.items():
    actual = spark.table(table).count() if spark.catalog.tableExists(table) else None
    status = "✅" if actual and (rows is None or actual == rows) else "❌"
    print(f"{status} {table}: {actual} wierszy" + (f" (oczekiwane {rows:,})" if rows else ""))
pdfs = sorted(p.name for p in Path(VOLUME_PATH).glob("*.pdf"))
print(f"{'✅' if len(pdfs) == 10 else '❌'} {VOLUME_PATH}: {len(pdfs)} PDF")

<!-- source: WS3[19] -->
## 2. AI Search: endpoint i indeks do skutku

U uczestników `00_setup` tylko uruchamia endpoint, a M3 czeka najwyżej kilka minut. Prowadzący potrzebuje gotowego indeksu przed M3, więc tu czekamy do końca (przy pierwszym uruchomieniu to najdłuższy krok przygotowania).

In [ ]:
# source: WS3[19] + new
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)
if not search_client.endpoint_exists(SEARCH_ENDPOINT):
    search_client.create_endpoint_and_wait(name=SEARCH_ENDPOINT, endpoint_type="STANDARD", verbose=True)
else:
    search_client.wait_for_endpoint(SEARCH_ENDPOINT, verbose=True)

if not search_client.index_exists(SEARCH_ENDPOINT, SEARCH_INDEX):
    search_client.create_delta_sync_index_and_wait(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=SEARCH_INDEX,
        primary_key="chunk_id",
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
        columns_to_sync=["doc_id", "filename", "chunk_position"],
        verbose=True,
    )
index = search_client.get_index(SEARCH_ENDPOINT, SEARCH_INDEX)
index.wait_until_ready(verbose=True)
status = index.describe().get("status", {})
print(f"✅ {SEARCH_INDEX}: ready={status.get('ready')} wierszy={status.get('indexed_row_count')}")

<!-- source: new + PRZ[19] -->
## 3–6. Funkcje, Genie, Knowledge Assistant, agent i aplikacja

1. **Funkcje UC:** `demo/m2_tool_calling` → **Run all** (tworzy `get_revenue_summary`, `get_average_customer_value`, `get_customer_profile`, `format_customer_for_agent`).
2. **Genie Agent:** kroki z `demo/m4_sql_genie_governance`, część 2. Tytuł dokładnie `Retail Customer Intelligence Assistant`. Na koniec M4 uruchom komórkę sprzątającą, żeby nie zostawić filtra i maski.
3. **Knowledge Assistant:** kroki z `demo/m3_rag_ai_search` (demo prowadzącego). Nazwę endpointu wpisz do `KA_ENDPOINT` w komórce kontekstu wyżej.
4. **Agent i aplikacja:** `demo/m5_end_to_end_agent` do komórki rejestracji `@champion` włącznie, potem kroki demo Databricks Apps (`sqlday-retail-agent`).
5. **Opcjonalnie:** `scripts/prepare_data_premium.ipynb`, krok 8 (bazowy wynik Genie do M4).

Rano w dniu warsztatu wystarczy komórka gotowości poniżej. Endpoint AI Search i aplikacja mogły się uśpić, a pierwsze zapytanie je budzi.

In [ ]:
# source: new + PRZ[19]
import mlflow
from mlflow import MlflowClient

checks = []


def check(name, fn):
    try:
        checks.append((name, True, fn()))
    except Exception as e:
        checks.append((name, False, f"{type(e).__name__}: {str(e)[:140]}"))


def gold():
    rows = spark.table(GOLD_TABLE).count()
    assert rows == 28_813, rows
    return f"{rows:,} wierszy, bez filtra"


def functions():
    names = {r["routine_name"] for r in spark.sql(f"SELECT routine_name FROM {CATALOG}.information_schema.routines WHERE routine_schema = '{SCHEMA}'").collect()}
    wanted = {"get_revenue_summary", "get_average_customer_value", "get_customer_profile", "format_customer_for_agent"}
    assert wanted <= names, wanted - names
    return "4 funkcje"


def no_governance_leftovers():
    detail = spark.sql(f"DESCRIBE TABLE EXTENDED {GOLD_TABLE}").toPandas().to_string().lower()
    assert "row filter" not in detail and "mask_tax_id" not in detail, "filtr lub maska z M4 nadal aktywne"
    return "brak filtra i maski"


def llm():
    client = w.serving_endpoints.get_open_ai_client()
    reply = client.chat.completions.create(model=LLM_ENDPOINT, messages=[{"role": "user", "content": "Odpowiedz: OK"}], max_tokens=5)
    return reply.choices[0].message.content.strip()


def embeddings():
    client = w.serving_endpoints.get_open_ai_client()
    return f"wymiar {len(client.embeddings.create(model=EMBEDDING_ENDPOINT, input=['test']).data[0].embedding)}"


def search():
    from databricks.ai_search.client import AISearchClient
    status = AISearchClient(disable_notice=True).get_index(SEARCH_ENDPOINT, SEARCH_INDEX).describe().get("status", {})
    assert status.get("ready"), status
    return f"indeks gotowy, {status.get('indexed_row_count')} wierszy"


def genie():
    space_id = next((s.space_id for s in (w.genie.list_spaces().spaces or []) if s.title == GENIE_TITLE), None)
    assert space_id, f"brak Genie Agenta „{GENIE_TITLE}”"
    return space_id


def mcp():
    import nest_asyncio
    from databricks_mcp import DatabricksMCPClient

    nest_asyncio.apply()  # list_tools() uruchamia asyncio.run, a notebook ma już pętlę zdarzeń
    tools = DatabricksMCPClient(server_url=f"{HOST}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}", workspace_client=w).list_tools()
    return f"{len(tools)} narzędzi z serwera funkcji"


def champion():
    version = MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(UC_MODEL_NAME, "champion")
    return f"{UC_MODEL_NAME} v{version.version}"


def app():
    state = w.apps.get(name=APP_NAME)
    return f"{state.compute_status.state if state.compute_status else '?'} | {state.url}"


def knowledge_assistant():
    assert KA_ENDPOINT, "uzupełnij KA_ENDPOINT"
    client = w.serving_endpoints.get_open_ai_client()
    return client.responses.create(model=KA_ENDPOINT, input=[{"role": "user", "content": "Ile mamy klientów VIP?"}]).output_text[:60]


def bakehouse():
    return f"{spark.table('samples.bakehouse.sales_transactions').count():,} transakcji"


def robotics_index():
    from databricks.ai_search.client import AISearchClient
    status = AISearchClient(disable_notice=True).get_index(SEARCH_ENDPOINT, f"{CATALOG}.{SCHEMA}.robotics_chunks_index").describe().get("status", {})
    assert status.get("ready") or str(status.get("detailed_state", "")).upper().startswith("ONLINE"), status
    return f"indeks robotyki gotowy, {status.get('indexed_row_count')} wierszy"


def experiment():
    return mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}").experiment_id


for name, fn in [
    ("gold_customer_360", gold), ("funkcje UC (M2)", functions), ("brak filtra/maski (M4)", no_governance_leftovers),
    ("model LLM", llm), ("model embeddingów", embeddings), ("AI Search (M3, M5)", search), ("Genie Agent (M4, M6)", genie),
    ("zarządzany MCP (M6)", mcp), ("agent @champion (M5)", champion), ("Databricks App (M5)", app),
    ("Knowledge Assistant (M3)", knowledge_assistant), ("eksperyment MLflow", experiment),
    ("Bakehouse (wzorce, capstone)", bakehouse), ("indeks robotyki (wzorzec M3)", robotics_index),
]:
    check(name, fn)

print(f"{'':2} {'Sprawdzenie':<28} Wynik")
for name, ok, detail in checks:
    print(f"{'✅' if ok else '❌'} {name:<28} {detail}")
print(f"\nGotowe: {sum(ok for _, ok, _ in checks)}/{len(checks)}")

<!-- source: new -->
## Próba na koncie Free Edition (raz przed warsztatem)

Na **osobnym koncie Free Edition** przejdź drogę uczestnika i zapisz czasy w `docs/rehearsal_log.md`:

1. Import repozytorium jako folder Git → `00_setup` Run all. Zanotuj, jak długo endpoint AI Search dochodzi do `ONLINE`.
2. `labs/m1` … `labs/m6` po kolei z rozwiązaniami skopiowanymi z `demo/`.
3. Wymuś każdy tryb awaryjny: `RUN_PARSE = False` (domyślnie), `SEARCH_READY = False` (uruchom M3 przed `ONLINE`), `TRY_MCP = False`.
4. Macierz tras w M5 trzy razy pod rząd: zanotuj, które trasy są niestabilne.